# myPolar – Traditional ML Baselines (SGD, SVM, Decision Tree, Random Forest)

**Goal:** classify Burmese sentences for the 3 POLAR subtasks.

| Task | Question | Type |
|---|---|---|
| Task 1 | Is the sentence polarized? | binary (0 / 1) |
| Task 2 | Who is the target? (political, racial/ethnic, religious, gender/sexual, other) | multi-label |
| Task 3 | How is polarization shown? (stereotype, vilification, ...) | multi-label |

**Plan:** load data → turn text into numbers → train 4 models → compare on *validate* → test the best model once on *test*.

## Step 1. Load the data
Each task has 3 files: **train** (to learn), **validate** (to compare models), **test** (final score, used only once).

In [11]:
%pip install scikit-learn

  Using cached scikit_learn-1.9.1-cp312-cp312-macosx_12_0_arm64.whl.metadata (9.3 kB)
  Using cached threadpoolctl-3.7.0-py3-none-any.whl.metadata (24 kB)
Using cached scikit_learn-1.9.1-cp312-cp312-macosx_12_0_arm64.whl (8.3 MB)
Using cached threadpoolctl-3.7.0-py3-none-any.whl (26 kB)
Note: you may need to restart the kernel to use updated packages.


In [19]:
import sklearn
print(sklearn.__version__)

1.9.1


In [5]:
import warnings
warnings.filterwarnings("ignore")      # hide warning messages to keep the output clean

import os
import pandas as pd

DATA_DIR = "data"                      # folder with our csv files
TEXT_COL = "text_syllable"             # text already split into syllables (space-separated)

# task name -> (file name prefix, label columns)
TASKS = {
    "task1": ("task1_polarization",  ["polarization"]),
    "task2": ("task2_type",          ["political", "racial_ethnic", "religious", "gender_sexual", "other"]),
    "task3": ("task3_manifestation", ["stereotype", "vilification", "dehumanization",
                                      "extreme_language", "lack_of_empathy", "invalidation"]),
}

# read train / dev / test for every task
data = {}
for task, (prefix, labels) in TASKS.items():
    data[task] = {}
    for split in ["train", "dev", "test"]:
        data[task][split] = pd.read_csv(f"{DATA_DIR}/{prefix}_{split}.csv", keep_default_na=False)
    print(f"{task}: train={len(data[task]['train'])}  dev={len(data[task]['dev'])}  test={len(data[task]['test'])}")

task1: train=7668  dev=1643  test=1644
task2: train=3570  dev=766  test=771
task3: train=3324  dev=710  test=709


Look at one example row:

In [7]:
data["task1"]["train"].head(3)

,id,text,text_syllable,polarization
0,mypolar_00001,မဟာသရေစည်သူဦးသန့်၂၂ဇန်နဝါရီ၁၉၀၉ – ၂၅နိုဝင်ဘာ၁၉...,မ ဟာ သ ရေ စည် သူ ဦး သ န့် ၂ ၂ ဇန် န ဝါ ရီ ၁ ၉ ...,0
1,mypolar_00007,မွေးချင်းညီအစ်ကိုလေးဦးအနက်အကြီးဆုံးဖြစ်သည်။,မွေး ချင်း ညီ အစ် ကို လေး ဦး အ နက် အ ကြီး ဆုံး...,0
2,mypolar_00008,မိဘများမှာအောင်မြင်သောမြေပိုင်ရှင်နှင့်ဆန်ကုန်...,မိ ဘ များ မှာ အောင် မြင် သော မြေ ပိုင် ရှင် နှ...,0


## Step 2. Turn text into numbers (features)
A model cannot read text, so we count words.
- **count**: how many times each syllable appears
- **tf-idf**: like count, but common syllables get a smaller weight
- **uni** = single syllables, **bi** = single syllables + pairs of syllables

⚠️ `token_pattern=r"\S+"` is important: it keeps every space-separated Burmese syllable as one token.
The default setting cuts Burmese into broken pieces.

In [21]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

FEATURES = {
    "uni_count": lambda: CountVectorizer(token_pattern=r"\S+", ngram_range=(1, 1)),
    "uni_tfidf": lambda: TfidfVectorizer(token_pattern=r"\S+", ngram_range=(1, 1), sublinear_tf=True),
    "bi_count":  lambda: CountVectorizer(token_pattern=r"\S+", ngram_range=(1, 2), min_df=2),   # min_df=2: ignore pairs seen only once
    "bi_tfidf":  lambda: TfidfVectorizer(token_pattern=r"\S+", ngram_range=(1, 2), min_df=2, sublinear_tf=True),
}

# check the tokens of the first sentence
example = data["task1"]["train"][TEXT_COL][0]
print("Sentence:", example[:60])
print("Tokens  :", FEATURES["uni_count"]().build_analyzer()(example)[:10])

Sentence: မ ဟာ သ ရေ စည် သူ ဦး သ န့် ၂ ၂ ဇန် န ဝါ ရီ ၁ ၉ ၀ ၉ – ၂ ၅ နို 
Tokens  : ['မ', 'ဟာ', 'သ', 'ရေ', 'စည်', 'သူ', 'ဦး', 'သ', 'န့်', '၂']


## Step 3. Choose the models
`class_weight="balanced"` gives more importance to rare labels (e.g. *religious*), so the model does not ignore them.

In [23]:
from sklearn.linear_model import SGDClassifier
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

MODELS = {
    "SGD":          lambda: SGDClassifier(loss="hinge", class_weight="balanced", random_state=42),
    "SVM":          lambda: LinearSVC(class_weight="balanced", random_state=42),
    "DecisionTree": lambda: DecisionTreeClassifier(class_weight="balanced", random_state=42),
    "RandomForest": lambda: RandomForestClassifier(n_estimators=200, class_weight="balanced",
                                                   n_jobs=-1, random_state=42),
}
print("Models:", list(MODELS))

Models: ['SGD', 'SVM', 'DecisionTree', 'RandomForest']


## Step 4. Train on *train*, compare on *dev*
- Task 1 has one label → one model.
- Task 2 and Task 3 have several labels → `OneVsRestClassifier` trains one small model per label.
- Score = **macro-F1** (average F1 over all labels, so rare labels count equally).

In [25]:
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import f1_score

def run(task, feat_name, model_name, eval_split="dev"):
    """Train one model on train, then predict eval_split ('dev' or 'test')."""
    prefix, labels = TASKS[task]
    train, evaluate = data[task]["train"], data[task][eval_split]

    # 1) text -> numbers (learn the vocabulary from train only)
    vectorizer = FEATURES[feat_name]()
    X_train = vectorizer.fit_transform(train[TEXT_COL])
    X_eval  = vectorizer.transform(evaluate[TEXT_COL])

    # 2) labels
    y_train = train[labels].values
    y_eval  = evaluate[labels].values

    # 3) model: one label -> normal model, many labels -> one model per label
    if len(labels) == 1:
        y_train, y_eval = y_train.ravel(), y_eval.ravel()
        model = MODELS[model_name]()
    else:
        model = OneVsRestClassifier(MODELS[model_name]())

    # 4) train and predict
    model.fit(X_train, y_train)
    y_pred = model.predict(X_eval)

    score = f1_score(y_eval, y_pred, average="macro", zero_division=0)
    return score, y_eval, y_pred


results = []                                   # we keep every score here
for task in TASKS:
    print(f"\n=== {task}: dev macro-F1 ===")
    for model_name in MODELS:
        for feat_name in FEATURES:
            score, _, _ = run(task, feat_name, model_name)
            results.append({"task": task, "model": model_name,
                            "features": feat_name, "dev_macro_f1": round(score, 4)})
            print(f"{model_name:<13} {feat_name:<10} {score:.4f}")


=== task1: dev macro-F1 ===
SGD           uni_count  0.7468
SGD           uni_tfidf  0.7588
SGD           bi_count   0.7687
SGD           bi_tfidf   0.7961
SVM           uni_count  0.7306
SVM           uni_tfidf  0.7620
SVM           bi_count   0.7607
SVM           bi_tfidf   0.7961
DecisionTree  uni_count  0.6689
DecisionTree  uni_tfidf  0.6427
DecisionTree  bi_count   0.6975
DecisionTree  bi_tfidf   0.6488
RandomForest  uni_count  0.7431
RandomForest  uni_tfidf  0.7443
RandomForest  bi_count   0.7492
RandomForest  bi_tfidf   0.7577

=== task2: dev macro-F1 ===
SGD           uni_count  0.5847
SGD           uni_tfidf  0.6175
SGD           bi_count   0.6345
SGD           bi_tfidf   0.6571
SVM           uni_count  0.5726
SVM           uni_tfidf  0.6205
SVM           bi_count   0.6101
SVM           bi_tfidf   0.6631
DecisionTree  uni_count  0.5205
DecisionTree  uni_tfidf  0.4783
DecisionTree  bi_count   0.5887
DecisionTree  bi_tfidf   0.5361
RandomForest  uni_count  0.5616
RandomForest  

SGD           bi_count   0.7687


SGD           bi_tfidf   0.7961


SVM           uni_count  0.7306
SVM           uni_tfidf  0.7620


SVM           bi_count   0.7607


SVM           bi_tfidf   0.7961


... (26 outputs left out)

RandomForest  uni_count  0.3767


RandomForest  uni_tfidf  0.3692


RandomForest  bi_count   0.3862


RandomForest  bi_tfidf   0.3761


## Step 5. Test the best model (only once per task)
We pick the model + features with the highest **dev** score, then check it on **test**.
We never choose a model by looking at test, so the test score is fair.

In [29]:
from sklearn.metrics import classification_report

results_df = pd.DataFrame(results)
final = []

for task in TASKS:
    # best row on dev for this task
    best = results_df[results_df.task == task].sort_values("dev_macro_f1", ascending=False).iloc[0]

    score, y_true, y_pred = run(task, best.features, best.model, eval_split="test")
    final.append({"task": task, "model": best.model, "features": best.features,
                  "dev_macro_f1": best.dev_macro_f1, "test_macro_f1": round(score, 4)})

    print(f"\n=== {task}: {best.model} + {best.features}  |  TEST macro-F1 = {score:.4f} ===")
    names = ["non-polarized", "polarized"] if task == "task1" else TASKS[task][1]
    print(classification_report(y_true, y_pred, target_names=names, digits=3, zero_division=0))


=== task1: SGD + bi_tfidf  |  TEST macro-F1 = 0.7841 ===
               precision    recall  f1-score   support

non-polarized      0.785     0.786     0.785       827
    polarized      0.783     0.782     0.783       817

     accuracy                          0.784      1644
    macro avg      0.784     0.784     0.784      1644
 weighted avg      0.784     0.784     0.784      1644


=== task2: SVM + bi_tfidf  |  TEST macro-F1 = 0.6624 ===
               precision    recall  f1-score   support

    political      0.798     0.784     0.791       388
racial_ethnic      0.639     0.608     0.623       125
    religious      0.778     0.538     0.636        65
gender_sexual      0.716     0.541     0.616        98
        other      0.670     0.624     0.646       250

    micro avg      0.732     0.674     0.702       926
    macro avg      0.720     0.619     0.662       926
 weighted avg      0.732     0.674     0.700       926
  samples avg      0.680     0.703     0.673       926

## Step 6. Summary tables (for the report)
Results are also saved in the `results` folder.

In [31]:
os.makedirs("results", exist_ok=True)

summary = pd.DataFrame(final)                                  # best model per task (test score)
table = results_df.pivot_table(index=["model", "features"],    # all dev scores in one table
                               columns="task", values="dev_macro_f1").round(3)

summary.to_csv("results/ml_test_results.csv", index=False)
table.to_csv("results/ml_dev_results.csv")

print("Best model per task:")
print(summary.to_string(index=False))
print("\nAll dev scores:")
print(table)

Best model per task:
 task model features  dev_macro_f1  test_macro_f1
task1   SGD bi_tfidf        0.7961         0.7841
task2   SVM bi_tfidf        0.6631         0.6624
task3   SVM bi_tfidf        0.5469         0.5122

All dev scores:
task                    task1  task2  task3
model        features                      
DecisionTree bi_count   0.698  0.589  0.472
             bi_tfidf   0.649  0.536  0.439
             uni_count  0.669  0.520  0.466
             uni_tfidf  0.643  0.478  0.455
RandomForest bi_count   0.749  0.649  0.532
             bi_tfidf   0.758  0.623  0.516
             uni_count  0.743  0.562  0.511
             uni_tfidf  0.744  0.591  0.514
SGD          bi_count   0.769  0.634  0.526
             bi_tfidf   0.796  0.657  0.531
             uni_count  0.747  0.585  0.480
             uni_tfidf  0.759  0.618  0.521
SVM          bi_count   0.761  0.610  0.503
             bi_tfidf   0.796  0.663  0.547
             uni_count  0.731  0.573  0.486
             u

## Results and findings

**Test macro-F1 of the best model per task (chosen on dev):**

| Task | Model | Features | Dev | Test |
|---|---|---|---|---|
| Task 1 — polarized? | SGD | bigram TF-IDF | 0.796 | **0.784** |
| Task 2 — target type | Linear SVM | bigram TF-IDF | 0.663 | **0.662** |
| Task 3 — manifestation | Linear SVM | bigram TF-IDF | 0.547 | **0.512** |

- Linear models (SGD, SVM) beat Decision Tree and Random Forest on all three tasks.
- Bigram TF-IDF was the best feature set.
- Keeping Burmese syllables whole (`token_pattern=r"\S+"`) raised Task 1 dev macro-F1 from 0.63 to 0.80 with the same SVM; the default scikit-learn tokenizer cut syllables into fragments.
- These models train in about one minute on a CPU, which makes them a strong baseline.